### Imports

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from scrapers import barbora, rimi
from scrapers.common import save_rows

### Barbora

In [ ]:
rows = barbora.scrape("c/LVOK/mesa-zuvis-ir-kulinarija/sviezia-mesa-ir-paukstiena", max_pages=6)
print(save_rows(rows, "barbora", "fresh-meat"))

### Rimi

#### First attempt

In [ ]:
rows = rimi.scrape("SH-9", "mesa-ir-zuvis-", max_pages=3)
print(save_rows(rows, "rimi", "meat-and-fish"))

#### With Discounts

In [ ]:
rows = rimi.scrape("SH-9", "mesa-ir-zuvis-", max_pages=3)

print(save_rows(rows, "rimi", "meat-and-fish"))

In [ ]:
rows = rimi.scrape(
    "SH-9",
    "mesa-ir-zuvis-",
    max_pages=3
)

print(save_rows(rows, "rimi", "meat-and-fish"))

In [ ]:
discounted = [p for p in rows if p["on_promo"]]

for p in discounted[:10]:
    print(
        p["name"],
        "| Regular:", p["regular_price"],
        "| Current:", p["current_price"],
        "| Promo:", p["promo_price"]
    )

In [ ]:
missing_prices = [
    p for p in rows
    if p["on_promo"] and p["promo_price"] is None
]

print("Missing promotional prices:", len(missing_prices))

for p in missing_prices:
    print(p["id"], p["name"])

##### Testing why discounts were missing

In [ ]:
discounted = [p for p in rows if p["on_promo"]]

print(f"Total products: {len(rows)}")
print(f"Discounted products: {len(discounted)}")

for p in discounted[:5]:
    print(
        p["name"],
        "| Original price:", p["price"],
        "| Promo price:", p["promo_price"],
        "| Discount:", p["discount_pct"],
        "| Rimi card:", p["requires_rimi_card"]
    )

In [ ]:
discounted[0]

In [ ]:
missing_prices = [
    p for p in rows
    if p["on_promo"] and p["promo_price"] is None
]

print(f"Promotions without extracted prices: {len(missing_prices)}")

for p in missing_prices[:5]:
    print(p["id"], p["name"], p["price"])

In [ ]:
target = missing_prices[0]

print(target["name"])
print(target["id"])

In [ ]:
from bs4 import BeautifulSoup
from scrapers.common import get_html

url = "https://www.rimi.lt/e-parduotuve/lt/produktai/mesa-ir-zuvis-/c/SH-9"

params = {
    "currentPage": 1,
    "pageSize": 40,
    "query": ":relevance:allCategories:SH-9:assortmentStatus:inAssortment"
}

html = get_html(url, params=params)
soup = BeautifulSoup(html, "html.parser")

product_div = soup.select_one(
    f'div[data-product-code="{target["id"]}"]'
)

card = product_div.find_parent("li") if product_div else None

print("Product found:", card is not None)

In [ ]:
if card:
    print("--- PRODUCT TEXT ---")
    print(card.get_text(" ", strip=True))

    print("\n--- PRICE LABEL HTML ---")
    price_label = card.select_one(".price-label")

    if price_label:
        print(price_label.prettify()[:4000])
    else:
        print("No .price-label found")

In [ ]:
import json

if card:
    info_tag = card.select_one("div[data-gtm-eec-product]")
    info = json.loads(info_tag["data-gtm-eec-product"])

    print("GTM price:", info["price"])
    print("GTM product data:", info)

In [ ]:
REGULAR_PRICE = re.compile(
    r"Įprasta kaina:\s*(\d+[.,]\d{2})\s*€"
)

text = card.get_text(" ", strip=True)

match = REGULAR_PRICE.search(text)

regular_price = (
    float(match.group(1).replace(",", "."))
    if match else None
)

print("Regular price:", regular_price)
print("Current price:", info["price"])

### Verify that it worked

In [ ]:
from scrapers.common import DATA_DIR
for f in sorted(DATA_DIR.rglob("*.json")):
    print(f.relative_to(DATA_DIR), len(pd.read_json(f)), "products")